#Importação das bibliotecas

In [6]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
import cv2
import pandas as pd

#https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html

#Importação do dataset

In [7]:
main_path = '256x256/'

Slices_Fraturados_path = os.path.join(main_path, 'Slices_Fraturados_PNG_256x256') #6717 imagens

Slices_Saudaveis_path = os.path.join(main_path, 'Slices_Saudaveis_PNG_256X256') #6717 imagens

In [8]:
#Criação do dataset

dataset = []

for filename_fraturadas in os.listdir(Slices_Fraturados_path):
    
    if filename_fraturadas.lower().endswith(('.png')):
        file_path = os.path.join(Slices_Fraturados_path, filename_fraturadas)
        dataset.append({'filepath': file_path, 'label': 1})

for filename_saudaveis in os.listdir(Slices_Saudaveis_path):
    
    if filename_saudaveis.lower().endswith(('.png')):
        file_path = os.path.join(Slices_Saudaveis_path, filename_saudaveis)
        dataset.append({'filepath': file_path, 'label': 0}) 

In [9]:
dataframe = pd.DataFrame(dataset)

In [10]:
print(len(dataframe))

print('\n')

print(dataframe['label'].value_counts())

13434


label
1    6717
0    6717
Name: count, dtype: int64


In [11]:
dataframe

,filepath,label
0,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1
1,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1
2,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1
3,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1
4,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1
...,...,...
13429,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0
13430,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0
13431,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0
13432,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0


In [12]:
dataframe['patient_id'] = dataframe['filepath'].apply(lambda x: os.path.basename(x).rsplit('_',1)[0])

In [13]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size = 0.2, random_state=42)

train_idx, val_idx = next(gss.split(dataframe, groups = dataframe['patient_id']))

train_dataframe = dataframe.iloc[train_idx].reset_index(drop=True)
val_dataframe = dataframe.iloc[val_idx].reset_index(drop=True)

In [14]:
pacientes_treino = set(train_dataframe['patient_id'])
pacientes_val = set(val_dataframe['patient_id'])
vazamento = pacientes_treino.intersection(pacientes_val)

print(f"Número de pacientes no Treino: {len(pacientes_treino)}")
print(f"Número de pacientes na Validação: {len(pacientes_val)}")
print(f"Pacientes em comum (Vazamento): {len(vazamento)}")

Número de pacientes no Treino: 932
Número de pacientes na Validação: 234
Pacientes em comum (Vazamento): 0


In [15]:
train_dataframe

,filepath,label,patient_id
0,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.27275
1,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.14360
2,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.24989
3,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.16838
4,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.5783
...,...,...,...
10980,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.3389
10981,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.28637
10982,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.16206
10983,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.2152


In [16]:
val_dataframe

,filepath,label,patient_id
0,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.7147
1,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.13509
2,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.9940
3,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.25772
4,256x256/Slices_Fraturados_PNG_256x256/1.2.826....,1,1.2.826.0.1.3680043.11901
...,...,...,...
2444,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.31569
2445,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.6892
2446,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.12182
2447,256x256/Slices_Saudaveis_PNG_256X256/1.2.826.0...,0,1.2.826.0.1.3680043.5949


In [17]:
print("--- Distribuição no Treino ---")
print(train_dataframe['label'].value_counts(normalize=True))

print("\n--- Distribuição na Validação ---")
print(val_dataframe['label'].value_counts(normalize=True))

--- Distribuição no Treino ---
label
1    0.507419
0    0.492581
Name: proportion, dtype: float64

--- Distribuição na Validação ---
label
0    0.533279
1    0.466721
Name: proportion, dtype: float64


In [18]:
train_dataframe.to_csv(os.path.join(main_path,'train_split.csv'), index=False)

val_dataframe.to_csv(os.path.join(main_path,'val_split.csv'), index=False)